# NSE オーナー企業フィルタリング

`nse_full_download.ipynb` が出力した CSV（`data/exports/nse/`）を読み込み、
**Promoter が自然人（本人・親族・取締役・KMP・ファミリートラスト）であり、
Promoter and Promoter Group の合計持分が 10% を超える銘柄**（＝オーナー企業）を
pandas だけで抽出する。

## 前提

- `nse_full_download.ipynb` の Phase 1〜4 + Cell 15（CSV エクスポート）が完了済み
- 必要な入力 CSV:
  - `data/exports/nse/stocks.csv`
  - `data/exports/nse/shareholdings.csv`
  - `data/exports/nse/shareholding_detail.csv`

## フィルタリングロジック（3 段階）

| Stage | 条件 | 根拠 |
|---|---|---|
| 1 | Promoter 合計持分 ≥ 10% | SEBI (SAST) 2011 Reg 3 substantial acquisition 閾値 |
| 2 | 自然人 sub-category 合計 > 0 | SEBI (ICDR) 2009 Reg 2(1)(zb) + BSE SHP XBRL taxonomy 2022-09-30 |
| 3 | 政府 sub-category 合計 < 0.5% | PSU（Public Sector Undertaking）を除外 |

出力: `data/exports/nse/owner_companies.csv`

In [ ]:
# Cell 2: Imports
from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

print("Imports OK")

In [ ]:
# Cell 3: Config
# ---------------------------------------------------------------------------
# 入力 CSV（nse_full_download.ipynb Cell 15 が出力済みの前提）
# ---------------------------------------------------------------------------
EXPORT_DIR: Path = Path("data/exports/nse")
STOCKS_CSV: Path = EXPORT_DIR / "stocks.csv"
SHAREHOLDINGS_CSV: Path = EXPORT_DIR / "shareholdings.csv"
SHAREHOLDING_DETAIL_CSV: Path = EXPORT_DIR / "shareholding_detail.csv"

# 出力先
OUTPUT_CSV: Path = EXPORT_DIR / "owner_companies.csv"

# ---------------------------------------------------------------------------
# フィルタ閾値（必要に応じて調整）
# ---------------------------------------------------------------------------
# Stage 1: Promoter 合計持分の最低ライン（%）
MIN_PROMOTER_PCT: float = 10.0

# Stage 2: 自然人シグナル合計の最低ライン（%）。0 にするとひとつでも非ゼロならヒット
MIN_NATURAL_SUM_PCT: float = 0.0

# Stage 3: 政府 promoter 合計の上限（%）。これを超えたら PSU として除外
MAX_GOVT_SUM_PCT: float = 0.5

print(f"EXPORT_DIR           = {EXPORT_DIR}")
print(f"MIN_PROMOTER_PCT     = {MIN_PROMOTER_PCT}%")
print(f"MIN_NATURAL_SUM_PCT  = {MIN_NATURAL_SUM_PCT}%")
print(f"MAX_GOVT_SUM_PCT     = {MAX_GOVT_SUM_PCT}%")
print(f"OUTPUT_CSV           = {OUTPUT_CSV}")

## データ読込

CSV を DataFrame に読み込む。`sub_category` は空文字（カテゴリ合計行）を持ちうるので
NaN を空文字に正規化する。

In [ ]:
# Cell 5: CSV 読込
stocks = pd.read_csv(STOCKS_CSV)
shareholdings = pd.read_csv(SHAREHOLDINGS_CSV)
detail = pd.read_csv(SHAREHOLDING_DETAIL_CSV)

# 空文字 sub_category は NaN 化されるので空文字に戻す（category 合計行の検出用）
detail["sub_category"] = detail["sub_category"].fillna("")
detail["pct_total_shares"] = pd.to_numeric(
    detail["pct_total_shares"], errors="coerce"
).fillna(0.0)

print(f"stocks              : {len(stocks):>8,} 行")
print(f"shareholdings       : {len(shareholdings):>8,} 行")
print(f"shareholding_detail : {len(detail):>8,} 行")
print(f"  - XBRL 取得済み銘柄: {detail['symbol'].nunique():>8,} 銘柄")

## Sub-category 集合の定義

SEBI (ICDR) 2009 Reg 2(1)(zb) + BSE SHP XBRL taxonomy 2022-09-30 / 2025-10-31 に準拠。

- **NATURAL_PERSON_SUBS**: 自然人（本人・親族・取締役・KMP・ファミリートラスト）が保有
- **GOVERNMENT_SUBS**: 中央・州・外国政府が promoter（PSU 除外用）
- **DOUBLE_COUNT_SUBS**: `Indian` / `Foreign` / `Governments` 等の**再集計行**。合計計算時は除外必須

In [ ]:
# Cell 7: Sub-category 定義

# 自然人（本人・親族・取締役・KMP・ファミリートラスト）
NATURAL_PERSON_SUBS: frozenset[str] = frozenset({
    "IndividualsOrHinduUndividedFamily",           # Table II A(1)(a)
    "NonResidentIndividualsOrForeignIndividuals",  # Table II A(2)(a)
    "DirectorsAndDirectorsRelatives",              # Table II A(1)(d) 内訳
    "KeyManagerialPersonnel",                      # Table II A(1)(d) 内訳
    "RelativesOfPromotersOtherThanPromoterGroup",  # Table II A(1)(d) 内訳
    "TrustsWhereAnyPersonBelongingToPromoterAndPromoterGroupIsisTrusteeOrBeneficiaryOrAuthorOfTrust",
})

# 政府 promoter（PSU 除外）
GOVERNMENT_SUBS: frozenset[str] = frozenset({
    "CentralGovernmentOrPresidentOfIndia",
    "StateGovernmentsOrGovernors",
    "ShareholdingByCompaniesOrBodiesCorporatewhereCentralOrStateGovernmentIsPromoter",
    "ForeignGovernment",
    "CentralGovernmentOrStateGovernmentS",  # 旧 taxonomy ラベル
})

# 再集計行（これらを SUM に含めると二重計上）
DOUBLE_COUNT_SUBS: frozenset[str] = frozenset({
    "Indian", "Foreign",
    "Goverments", "Governments", "CentralAndStateGovernments",
})

print(f"自然人 sub-category  : {len(NATURAL_PERSON_SUBS)} 種")
print(f"政府 sub-category    : {len(GOVERNMENT_SUBS)} 種")
print(f"再集計（除外）       : {len(DOUBLE_COUNT_SUBS)} 種")

## Stage 1〜3 フィルタ適用

1. Promoter カテゴリの **集計行のみ**（`is_category_total=1`）を抽出
2. 銘柄ごとに **最新 report_date** のスナップショットに絞る
3. カテゴリ合計行（`sub_category=''`）を `promoter_total_pct` とする
4. 自然人 / 政府 sub-category を `isin()` で集計
5. 3 条件すべて満たす銘柄だけ残す

In [ ]:
# Cell 9: Stage 1-3 フィルタ適用

# Promoter カテゴリの集計行のみ抽出
promoter = detail[
    (detail["category"] == "PromoterAndPromoterGroup")
    & (detail["is_category_total"] == 1)
].copy()

# 銘柄ごとに最新 report_date（YYYY-MM-DD ISO 形式なので文字列比較で OK）
latest = (
    promoter.groupby("symbol")["report_date"].max().rename("latest_report_date")
)
promoter = promoter.merge(latest, on="symbol")
promoter = promoter[promoter["report_date"] == promoter["latest_report_date"]]

# Promoter 合計（sub_category='' のカテゴリ合計行）
promoter_total = (
    promoter[promoter["sub_category"] == ""]
    .groupby("symbol")["pct_total_shares"]
    .first()
    .rename("promoter_total_pct")
)

# 自然人 sub-category の合計
natural_sum = (
    promoter[promoter["sub_category"].isin(NATURAL_PERSON_SUBS)]
    .groupby("symbol")["pct_total_shares"]
    .sum()
    .rename("natural_sum_pct")
)

# 政府 sub-category の合計
govt_sum = (
    promoter[promoter["sub_category"].isin(GOVERNMENT_SUBS)]
    .groupby("symbol")["pct_total_shares"]
    .sum()
    .rename("govt_sum_pct")
)

# 最新 report_date
report_date = (
    promoter.groupby("symbol")["report_date"].first().rename("report_date")
)

# 結合（0 埋めで sub-category 未報告銘柄も残す）
aggregated = pd.concat(
    [promoter_total, natural_sum, govt_sum, report_date],
    axis=1,
).fillna({
    "promoter_total_pct": 0.0,
    "natural_sum_pct": 0.0,
    "govt_sum_pct": 0.0,
})

# ---------------------------------------------------------------------------
# 3 段階フィルタ
# ---------------------------------------------------------------------------
mask_stage1 = aggregated["promoter_total_pct"] >= MIN_PROMOTER_PCT
mask_stage2 = aggregated["natural_sum_pct"] > MIN_NATURAL_SUM_PCT
mask_stage3 = aggregated["govt_sum_pct"] < MAX_GOVT_SUM_PCT

owners = aggregated[mask_stage1 & mask_stage2 & mask_stage3].copy()
owners["natural_ratio"] = owners["natural_sum_pct"] / owners["promoter_total_pct"]

# stocks テーブルから会社情報を結合
meta_cols = [c for c in ["symbol", "company_name", "sector", "industry", "is_fno"] if c in stocks.columns]
owners = (
    owners.reset_index()
    .merge(stocks[meta_cols], on="symbol", how="left")
    .set_index("symbol")
    .sort_values("promoter_total_pct", ascending=False)
)

print(f"総 XBRL 取得済み銘柄         : {len(aggregated):>6,}")
print(f"Stage 1 (promoter >= {MIN_PROMOTER_PCT}%) : {mask_stage1.sum():>6,}")
print(f"Stage 2 (自然人 > {MIN_NATURAL_SUM_PCT}%)       : {(mask_stage1 & mask_stage2).sum():>6,}")
print(f"Stage 3 (政府 < {MAX_GOVT_SUM_PCT}%)        : {len(owners):>6,}  ← オーナー企業")

## 結果確認・CSV 出力

In [ ]:
# Cell 11: 結果を CSV 出力

export_cols = [
    "company_name", "sector", "industry",
    "report_date",
    "promoter_total_pct", "natural_sum_pct", "govt_sum_pct", "natural_ratio",
]
export_cols = [c for c in export_cols if c in owners.columns]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
owners.reset_index()[["symbol", *export_cols]].to_csv(
    OUTPUT_CSV, index=False, encoding="utf-8-sig"
)
print(f"オーナー企業 {len(owners):,} 件を出力 → {OUTPUT_CSV}")

# 上位 30 件表示
print("\n--- Promoter 合計保有率 上位 30 ---")
display(owners[export_cols].head(30))  # noqa: F821

## 分布・内訳分析

In [ ]:
# Cell 13: 分布・内訳分析

print("--- Promoter 合計保有率の分布 ---")
display(owners["promoter_total_pct"].describe())  # noqa: F821

print("\n--- natural_ratio（自然人直接保有 ÷ Promoter合計）分布 ---")
print("  ※ 低い = holding company 経由型、高い = 直接保有型")
display(owners["natural_ratio"].describe())  # noqa: F821

if "sector" in owners.columns:
    print("\n--- セクター別オーナー企業数（上位 15）---")
    display(  # noqa: F821
        owners.groupby("sector", dropna=False)
        .size()
        .sort_values(ascending=False)
        .head(15)
        .rename("count")
    )

# 2 類型の可視化
print("\n--- 2 類型別銘柄数 ---")
direct_type = owners[owners["natural_ratio"] >= 0.5]
holding_type = owners[owners["natural_ratio"] < 0.5]
print(f"  直接保有型     (natural_ratio >= 0.5): {len(direct_type):>5,} 銘柄")
print(f"  holding 経由型 (natural_ratio <  0.5): {len(holding_type):>5,} 銘柄")

## サンプル銘柄の内訳検証

RELIANCE などの代表銘柄について、Promoter カテゴリ内の sub-category 別内訳を確認する。

In [ ]:
# Cell 15: サンプル銘柄の内訳検証

SAMPLE_SYMBOLS: list[str] = ["RELIANCE", "INFY", "TCS", "BAJFINANCE", "ASIANPAINT"]

def show_breakdown(symbol: str) -> None:
    if symbol not in owners.index:
        print(f"[{symbol}] オーナー企業リストに含まれない（Stage 1-3 で除外）")
        return
    rd = owners.loc[symbol, "report_date"]
    breakdown = detail[
        (detail["symbol"] == symbol)
        & (detail["category"] == "PromoterAndPromoterGroup")
        & (detail["is_category_total"] == 1)
        & (detail["report_date"] == rd)
        & (~detail["sub_category"].isin(DOUBLE_COUNT_SUBS))
    ].sort_values("pct_total_shares", ascending=False)
    print(f"\n=== {symbol} ({rd}) ===")
    print(
        f"  promoter_total={owners.loc[symbol, 'promoter_total_pct']:.2f}%  "
        f"natural={owners.loc[symbol, 'natural_sum_pct']:.2f}%  "
        f"govt={owners.loc[symbol, 'govt_sum_pct']:.2f}%  "
        f"ratio={owners.loc[symbol, 'natural_ratio']:.3f}"
    )
    display(  # noqa: F821
        breakdown[["sub_category", "pct_total_shares", "num_shareholders"]]
        .reset_index(drop=True)
    )

for sym in SAMPLE_SYMBOLS:
    show_breakdown(sym)

## 除外された銘柄の確認（デバッグ用）

Stage 2 または Stage 3 で除外された上位銘柄を表示し、フィルタが意図通り動作しているか確認する。

期待: LICI / IDBI / IOB / UCOBANK / CENTRALBK 等の PSU が Stage 3 で除外されていること。

In [ ]:
# Cell 17: 除外銘柄の確認

excluded = aggregated[mask_stage1 & ~(mask_stage2 & mask_stage3)].copy()
excluded = excluded.reset_index().merge(
    stocks[[c for c in ["symbol", "company_name"] if c in stocks.columns]],
    on="symbol",
    how="left",
)
excluded["reason"] = ""
excluded.loc[excluded["govt_sum_pct"] >= MAX_GOVT_SUM_PCT, "reason"] = "Stage3: 政府 promoter"
excluded.loc[
    (excluded["natural_sum_pct"] <= MIN_NATURAL_SUM_PCT) & (excluded["reason"] == ""),
    "reason",
] = "Stage2: 自然人シグナルなし"

print(f"Stage 1 通過後に除外された銘柄: {len(excluded):,}")
print("\n--- 除外銘柄 上位 20（promoter_total_pct 降順）---")
display(  # noqa: F821
    excluded.sort_values("promoter_total_pct", ascending=False)
    .head(20)[
        ["symbol", "company_name", "promoter_total_pct",
         "natural_sum_pct", "govt_sum_pct", "reason"]
    ]
)